# 01 — Cleaning the Lending Club dump

This notebook is step one of a credit-risk project I'm building while working through Andrew Ng's ML Specialization. The goal here is narrow: take the raw Lending Club CSV (2007–2018, ~2.26M rows, 145 columns) and produce a smaller, typed, sampled file that notebooks 02 (EDA) and 03 (feature engineering / modeling prep) can use.

I'm deliberately **not** doing feature engineering here. Anything that creates a new feature from existing ones — ratios, encodings, scaling, imputation — lives in `03_feature_engineering.ipynb` so it happens *after* the train/test split. That's the "fit preprocessing on train only" rule from the course, and it's easier to respect if preprocessing lives in one place rather than being sprinkled across every notebook.

In [13]:
%pip install pyarrow 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 30.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)

## Loading the raw file

The raw CSV is ~1.5 GB and has mixed-type columns that trip pandas' default `low_memory=True` with cryptic dtype warnings. Explicit `low_memory=False` is what works.

In [17]:
df = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", engine="pyarrow")
df.shape

(2260701, 151)

In [18]:
df.head(3)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,...,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,truck driver,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,home_improvement,NaN,605xx,IL,10.78,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## What's actually in `loan_status`?

Before deciding what the target is, I want to see every status value and roughly how common it is.

In [4]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

## Which columns to keep, and which to drop as leakage

Lending Club gives you 145 columns, but most fall into one of three unhelpful piles:

1. **Identifiers** (`id`, `member_id`, `url`) — no predictive value, just row keys.
2. **Post-origination columns** — things like `total_pymnt`, `recoveries`, `last_pymnt_amnt`, `collection_recovery_fee`. These only exist *after* the loan has been serviced for some time, so they trivially predict the outcome. The lecture on data leakage in Course 2 is specifically about this situation: if the feature wouldn't have been available at prediction time (loan origination), it can't be in the model.
3. **Columns with mostly missing values** (>40% missing in the full set) — things like `desc` and the various `*_joint` columns. Not worth the trouble on this first pass.

What's left is 32 columns I believe could plausibly be known on the day a loan is issued. I'm being deliberately conservative: if I'm unsure whether a column is pre- or post-origination, I'm dropping it.

In [5]:
keep_cols = [
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "purpose", "addr_state", "dti",
    "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "mths_since_last_delinq", "mths_since_last_record",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "collections_12_mths_ex_med", "acc_now_delinq", "tot_coll_amt",
    "tot_cur_bal", "total_rev_hi_lim",
    "loan_status", "issue_d",
]

keep_cols = [c for c in keep_cols if c in df.columns]
df_clean = df[keep_cols].copy()
df_clean.shape

(2260701, 32)

## Defining the target — what counts as a "default"?

`loan_status` has about ten possible values. I need to collapse them to a binary target: did this loan default or not? Two sub-decisions:

1. **Which statuses count as the positive class.** I'm grouping `Charged Off`, `Default`, `Late (31–120 days)`, `Late (16–30 days)` and the two `Does not meet the credit policy...Charged Off` variants as positive (1 = defaulted). Including the "late" statuses is a judgment call — those borrowers aren't technically defaults yet, they're currently in trouble. For a credit team's operational decision ("is this loan showing distress?") they belong on the same side. I'll check whether this biases probability calibration in notebook 04.
2. **Dropping `Current` loans.** Those are still being paid; we don't know their final outcome. I don't want to make up a label for them, so they come out of the dataset entirely.

In [6]:
default_statuses = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)",
    "Does not meet the credit policy. Status:Charged Off",
]

completed_statuses = default_statuses + [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
]

df_clean = df_clean[df_clean["loan_status"].isin(completed_statuses)].copy()
df_clean["target"] = df_clean["loan_status"].isin(default_statuses).astype(int)

print(f"rows after filtering: {len(df_clean):,}")
print(f"default rate:         {df_clean['target'].mean()*100:.2f}%")

rows after filtering: 1,373,915
default rate:         21.48%


## Sampling down to 250k rows

The full completed-loan set is around 1.3M rows. I can fit that in memory, but iterating on EDA and modeling gets slow. I'm stratified-sampling 250k rows to keep the class ratio stable while shrinking everything else.

What I lose: the long tail of rare states and loan purposes gets thinner, so a couple of charts in notebook 02 will be noisier for low-volume categories. Worth it for interactivity.

In [7]:
SAMPLE = 250_000

if len(df_clean) > SAMPLE:
    df_clean, _ = train_test_split(
        df_clean,
        train_size=SAMPLE,
        stratify=df_clean["target"],
        random_state=42,
    )

print(f"after sampling: {df_clean.shape}")
print(f"default rate:   {df_clean['target'].mean()*100:.2f}%")

after sampling: (250000, 33)
default rate:   21.48%


## Converting a few columns to the right types

Three columns come in as strings but are logically numeric (`term`, `int_rate`, `revol_util`), and `earliest_cr_line` + `issue_d` come in as `MMM-YYYY` strings that I need as real dates — `issue_d` for the time-aware split in notebook 03, and `earliest_cr_line` so 03 can compute credit history as `(issue_d − earliest_cr_line)` per row (an as-of calculation, not a hard-coded reference date).

In [8]:
df_clean["term"] = df_clean["term"].str.extract(r"(\d+)").astype(float)

for col in ["int_rate", "revol_util"]:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].str.replace("%", "", regex=False).astype(float)

# Lending Club uses a 4-digit year, so the format is %b-%Y, not %b-%y.
# Passing %b-%y makes pandas silently misparse "Aug-2003" into the 20YY range.
df_clean["earliest_cr_line_dt"] = pd.to_datetime(
    df_clean["earliest_cr_line"], format="%b-%Y", errors="coerce"
)
df_clean["issue_d_dt"] = pd.to_datetime(
    df_clean["issue_d"], format="%b-%Y", errors="coerce"
)

n_bad = df_clean["earliest_cr_line_dt"].isna().sum()
print(f"earliest_cr_line unparseable: {n_bad:,} of {len(df_clean):,}")

earliest_cr_line unparseable: 5 of 250,000


## How bad is the missingness?

Just the summary — I'm not imputing in this notebook (next section explains why).

In [9]:
missing = df_clean.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df_clean) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct})

,missing,pct
mths_since_last_record,207463,82.99
mths_since_last_delinq,125618,50.25
emp_length,14737,5.89
total_rev_hi_lim,12883,5.15
tot_cur_bal,12883,5.15
tot_coll_amt,12883,5.15
revol_util,164,0.07
dti,85,0.03
collections_12_mths_ex_med,30,0.01
pub_rec,5,0.00


## Why I'm not imputing in this notebook

Earlier I had `df_clean.fillna(df_clean.median())` sitting right here and it was wrong. Any preprocessing step that fits parameters from the data — a median, a mean, a scaler, a target encoder — has to be fit on the training set *after* the train/test split, or information from the test set leaks into training through those parameters. The specialization makes this explicit in the lecture on applying ML; the practical consequence is that validation metrics look better than production performance, which is exactly the gap you don't want to discover after shipping.

So notebook 01 saves the data **with** its NaN values intact. Notebook 03 does the time-aware split by `issue_d` first, then fits the imputer (plus any scaler / encoder) on the train slice only and applies it to both slices.

Notebook 02's EDA tolerates NaN — `seaborn.histplot` and `pandas.groupby` ignore missing rows by default. I'll flag per-chart where the missing-row count matters.

## What the cleaned dataset looks like

In [10]:
print(f"shape:        {df_clean.shape}")
print(f"default rate: {df_clean['target'].mean()*100:.2f}%")
print(f"date range:   {df_clean['issue_d_dt'].min():%Y-%m} to {df_clean['issue_d_dt'].max():%Y-%m}")

shape:        (250000, 35)
default rate: 21.48%
date range:   2007-06 to 2018-12


In [11]:
df_clean.describe()

,loan_amnt,term,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,target,earliest_cr_line_dt,issue_d_dt
count,250000.000000,250000.000000,250000.000000,250000.000000,2.500000e+05,249915.000000,249995.000000,250000.000000,250000.000000,249995.000000,124382.000000,42537.000000,249995.000000,249995.000000,2.500000e+05,249836.000000,249995.000000,249970.000000,249995.000000,2.371170e+05,2.371170e+05,2.371170e+05,250000.000000,249995,250000
mean,14476.741900,41.842560,13.289079,439.770946,7.625523e+04,18.291976,0.318382,696.051240,700.051372,0.662073,34.322346,70.339704,11.583580,0.215604,1.634151e+04,51.772891,24.934835,0.017050,0.005160,2.657597e+02,1.411978e+05,3.285919e+04,0.214844,1999-03-16 11:59:35.289105,2015-06-17 04:53:34.540800
min,900.000000,36.000000,5.310000,21.250000,0.000000e+00,0.000000,0.000000,630.000000,634.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,1944-01-01 00:00:00,2007-06-01 00:00:00
25%,8000.000000,36.000000,9.750000,249.080000,4.600000e+04,11.800000,0.000000,670.000000,674.000000,0.000000,16.000000,53.000000,8.000000,0.000000,5.916000e+03,33.400000,16.000000,0.000000,0.000000,0.000000e+00,2.937600e+04,1.400000e+04,0.000000,1995-05-01 00:00:00,2014-07-01 00:00:00
50%,12000.000000,36.000000,12.790000,375.490000,6.500000e+04,17.630000,0.000000,690.000000,694.000000,0.000000,31.000000,71.000000,11.000000,0.000000,1.111800e+04,52.100000,23.000000,0.000000,0.000000,0.000000e+00,8.020700e+04,2.410000e+04,0.000000,2000-08-01 00:00:00,2015-08-01 00:00:00
75%,20000.000000,36.000000,16.020000,583.382500,9.000000e+04,24.090000,0.000000,710.000000,714.000000,1.000000,50.000000,90.000000,14.000000,0.000000,1.978025e+04,70.700000,32.000000,0.000000,0.000000,0.000000e+00,2.106940e+05,4.070000e+04,0.000000,2004-07-01 00:00:00,2016-08-01 00:00:00
max,40000.000000,60.000000,30.990000,1691.280000,9.522972e+06,999.000000,21.000000,845.000000,850.000000,31.000000,180.000000,124.000000,79.000000,49.000000,2.560703e+06,193.000000,156.000000,9.000000,6.000000,6.214661e+06,4.772549e+06,9.999999e+06,1.000000,2015-09-01 00:00:00,2018-12-01 00:00:00
std,8755.644942,10.299823,4.787245,262.706835,6.120769e+04,11.192252,0.875505,31.772112,31.772733,0.965127,21.958044,26.878922,5.483347,0.597652,2.325751e+04,24.542813,12.004040,0.142604,0.077701,1.304333e+04,1.587335e+05,3.958811e+04,0.410715,NaN,NaN


## Save for the next notebook

In [12]:
out_path = "../data/processed/loan_cleaned.csv"
df_clean.to_csv(out_path, index=False)
print(f"wrote {len(df_clean):,} rows to {out_path}")

wrote 250,000 rows to ../data/processed/loan_cleaned.csv


## One thing that surprised me, one thing I'd do differently

**Surprised me:** how much of this file is leakage. Roughly 40 of the 145 raw columns are only populated after the loan has been serviced — payment histories, recovery amounts, hardship flags. A model that accidentally keeps even one of them (say, `last_pymnt_amnt`) will look amazing in validation and die in production. The leakage lecture made sense in the abstract; this was the first time I had to apply the heuristic column by column on something that wasn't already cleaned for me.

**Would do differently:** stop treating "cleaning" and "feature engineering" as one continuous flow. The first version of this notebook had both mixed together, with imputation and derived features all in one pass, and that's exactly how the median-leakage bug happened — once imputation and FE are entangled, it feels natural to do them both *before* the split. Keeping them in separate notebooks, with the split in between, is a cheap way to make that mistake harder to make.